## 0. Prerequisites

1. **Full Disk Access** — grant it to the app running this notebook (System Settings → Privacy & Security → Full Disk Access), or the agent can't read files under `~/Documents`.
2. **LibreOffice** — installed at `/Applications/LibreOffice.app` (renders `.docx`/`.pptx`).
3. **Ollama model** (only needed for the agent section): `ollama pull qwen3:8b`.

Select the **Python (nus-module-agent)** kernel (top-right).

# NUS Module Search Agent

Takes a fuzzy query, finds matching course files (PDF / Word / PowerPoint), ranks them, and renders up to **3 page/slide images per file** so you can visually confirm the match.

The same retrieval + rendering pipeline is driven by **two interchangeable agent designs**, both shown below:

- **Single agent (LangChain)** — one `create_agent` tool-calling loop (§7a).
- **Multi-agent graph (LangGraph)** — a *supervisor* routing specialist nodes, with a grader → re-query reflection loop (§7b).

This notebook is built to *teach* each piece, then run them together.

```
                                  the AGENT(S) decide which tools to call
                                  │
  query ─► search_courses ─► [ embeddings ─► Chroma ─► rerank ─► aggregate by file/page ]
          render_snapshots ─► [ PyMuPDF renders the matching pages to PNG ]
```

**Roles**
- *Embeddings* (sentence-transformers, local): turn text into meaning-vectors.
- *Chroma* (local, persistent): store vectors + find the closest ones.
- *Reranker* (cross-encoder, local): sharpen the top matches.
- *LLM* (Ollama now; Claude/OpenAI later): orchestrates the tools = the *agent*.
- *PyMuPDF + LibreOffice*: render the page/slide you'll actually look at.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import config
print('Data root :', config.DATA_ROOT)
print('Embed model:', config.EMBED_MODEL)
print('Reranker  :', config.RERANK_MODEL or '(disabled)')
print('LLM       :', config.LLM_PROVIDER, '/', config.LLM_MODEL)

## 1. Embeddings — text becomes a meaning-vector

An embedding maps text to a vector so that *similar meaning → similar direction*. This is what makes search **fuzzy**: the query and the slide can share zero words yet still match. Below, the paraphrase should score far higher than the unrelated sentence — with no shared keywords.

In [ ]:
import numpy as np
from src.index import get_embeddings

emb = get_embeddings()  # downloads the model on first run

def cos(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

query   = emb.embed_query('how cloud vendors bill you for servers')
para    = emb.embed_query('pricing models for on-demand virtual machine instances')
unrel   = emb.embed_query('the silhouette score evaluates clustering quality')

print(f'query vs paraphrase  : {cos(query, para):.3f}   <- high (same meaning, no shared words)')
print(f'query vs unrelated   : {cos(query, unrel):.3f}   <- low')
print('vector length        :', len(query))

## 2. Build the index (Chroma)

`build_index()` walks your course folders, converts Office files to PDF (cached), splits each page into chunks, embeds them, and stores the vectors in a **persistent** Chroma DB. It's **incremental** — re-running only processes new or changed files.

In [ ]:
from src import index

# First run can take a while (Office conversion + embedding every page).
summary = index.build_index()
summary

## 3. Retrieval + ranking — chunks → files → pages

Similarity is computed **query ↔ chunk**. We then roll those chunk scores up:
- **file score = its single best chunk** (max-pool) → rank files, best on top;
- **per file, the pages behind the top chunks** → the pages to snapshot (up to 3).

In [ ]:
from src import retrieve

QUERY = 'how do cloud providers charge for virtual machines'
results = retrieve.search(QUERY)

for r in results:
    pages = ', '.join(str(p['page']) for p in r['pages'])
    print(f"{r['rank']}. {r['file_name']}  [{r['module']}]  score={r['score']:.3f}  pages: {pages}")
    for p in r['pages']:
        print(f"      p{p['page']}: {p['snippet'][:120]}")

## 4. Snapshots — see the matching pages

`search_and_render` renders the top pages of each file to PNG so you can confirm at a glance.

In [ ]:
from IPython.display import Image, display, Markdown

rendered = retrieve.search_and_render(QUERY, max_files=3)
for r in rendered:
    display(Markdown(f"### {r['rank']}. {r['file_name']}  ·  *{r['module']}*  ·  score {r['score']:.2f}"))
    for s in r['snapshots']:
        display(Markdown(f"page {s['page']}"))
        display(Image(filename=s['image'], width=520))

## 5. The pluggable LLM

`get_llm()` returns a LangChain chat model. Local Ollama is the default; switch to Claude or OpenAI by passing `provider=` (or setting `NUS_LLM_PROVIDER`). The agent code never changes.

In [ ]:
from src.llm import get_llm

try:
    llm = get_llm()  # provider='anthropic' or 'openai' to switch later
    print(llm.invoke('Reply with exactly: ready').content)
except Exception as e:
    print('LLM not ready (pull a model first: `ollama pull qwen3:8b`).')
    print(type(e).__name__, e)

## 6. Tools — what the agent is allowed to do

A LangChain *tool* is a plain function the LLM can call. Ours wrap the deterministic pipeline. You can call them directly (no LLM) to see their text output — this is exactly what the agent sees. **Both** agent designs in §7 use these same two tools.

In [ ]:
from src.agent import search_courses, render_file_snapshots

print(search_courses.invoke({'query': QUERY})[:800])

## 7. The agent — two designs

The tools above are wrapped by two interchangeable orchestrations over the *same* pipeline. We run both so you can compare them.

### 7a. Single agent (LangChain)

`create_agent(llm, tools, system_prompt)` (LangChain 1.x) builds **one** tool-calling agent on top of LangGraph. It runs an **iterative loop**: the LLM picks a tool, sees the result, and decides what to do next, repeating until it stops calling tools and writes its answer. So for a fuzzy request it typically calls `search_courses` (possibly more than once, reformulating if needed), then `render_file_snapshots`, then summarises — but the whole control flow is *one LLM following one prompt*, with no separate, guaranteed evaluation step. Requires an Ollama model.

In [ ]:
from src import agent as agent_mod
from langchain_core.messages import ToolMessage
from IPython.display import Image, Markdown, display

agent = agent_mod.build_agent()
out = agent.invoke({"messages": [{"role": "user",
                                   "content": "I need the slides about how the cloud charges for compute"}]})

display(Markdown(out["messages"][-1].content))      # the LLM's text answer

# Pull image paths out of every render_file_snapshots ToolMessage in this turn:
for m in out["messages"]:
    if isinstance(m, ToolMessage) and m.name == "render_file_snapshots":
        current_file = None
        for line in m.content.splitlines():
            stripped = line.strip()
            if stripped.startswith("Rendered ") and " for " in stripped:
                # header line: "Rendered N image(s) for FILE_NAME:"
                current_file = stripped.split(" for ", 1)[1].rstrip(":")
            elif stripped.startswith("p") and stripped.endswith(".png"):
                page, path = (s.strip() for s in stripped.split(":", 1))
                display(Markdown(f"**{current_file}** — {page}  \n`{path}`"))
                display(Image(filename=path, width=560))

### 7b. Multi-agent graph (LangGraph)

Instead of one agent, [`src/graph.py`](../src/graph.py) defines a **supervisor** that routes between specialist nodes:

- **query** — turns the fuzzy request into concrete search strings
- **retrieval** — runs the search *(deterministic)*
- **grader** — judges whether the hits actually match; if **weak**, the supervisor loops back to **query** to reformulate (a reflection loop, bounded by `graph.MAX_QUERY_ATTEMPTS`)
- **render** — snapshots the top pages *(deterministic)*
- **synthesize** — writes the final ranked answer

Both designs *can* search more than once — in §7a that's up to the LLM's own tool-calling loop. The difference is **where the control lives**: here reflection is **explicit and enforced** — a dedicated `grader` always evaluates the results, and the re-query loop is a structured, bounded, inspectable step rather than behaviour you hope the LLM happens to exhibit. Each node also gets its own prompt (and can use its own model). `run()` returns the full shared state; the cell below uses `.stream()` so you can watch the supervisor route, node by node.

In [ ]:
from src.graph import build_graph
from IPython.display import Image, display, Markdown

app = build_graph()
state = {}   # accumulate the streamed updates into the final shared state

print("--- supervisor routing ---")
for step in app.stream(
    {"user_request": "find the slide that illustrates how a transformer encoder works",
     "attempts": 0},
    config={"recursion_limit": 25},
):
    for node, update in step.items():
        state.update(update)
        if node == "supervisor":  print("supervisor ->", update.get("route"))
        elif node == "query":     print("query      ", update.get("queries"))
        elif node == "retrieval": print("retrieval  ", len(update.get("hits", [])), "files")
        elif node == "grader":    print("grader     ", update.get("grade"), "—", update.get("grade_reason"))

display(Markdown("### Final answer\n" + (state.get("answer") or "*(none)*")))

# Pages the graph rendered (empty if Full Disk Access blocks the source files):
for snap in state.get("snapshots", []):
    for img in snap.get("images", []):
        display(Markdown(f"**{snap['file_name']}** — p{img['page']}"))
        display(Image(filename=img["image"], width=560))

## 8. Notes

- **Two designs, one pipeline**: §7a (single LangChain agent) and §7b (LangGraph supervisor) call the *same* tools and retrieval/render code — only the orchestration differs.
- **Watch the graph route**: stream it (as in §7b) or run `python tests/test_graph_smoke.py`. Visualize the topology with `build_graph().get_graph().draw_mermaid_png()`.
- **Reflection loop bound**: `graph.MAX_QUERY_ATTEMPTS` caps how many times the grader can send the query agent back to reformulate.
- **Swap the LLM**: `build_agent(provider='anthropic')` or set `NUS_LLM_PROVIDER=anthropic` + `ANTHROPIC_API_KEY`.
- **Swap embeddings**: change `NUS_EMBED_MODEL` and re-index (`index.build_index(force=True)`); different models produce incompatible vectors.
- **Re-index after adding files**: just call `index.build_index()` again — only new/changed files are processed.
- **Tuning**: `config.MAX_FILES`, `config.MAX_PAGES_PER_FILE`, `config.TOP_K_CHUNKS`, and `config.RENDER_DPI`.